<a href="https://colab.research.google.com/github/bemakerorg/AIoT_Book_II_RF/blob/main/AIoT_RF_Book_ES_19.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Esercizio 19 - Cani contro Gatti**


In [ ]:
# 1) Importiamo le librerie necessarie
import tensorflow as tf              # TensorFlow: framework per l'apprendimento automatico e il deep learning
import tensorflow_hub as hub         # TensorFlow Hub: per utilizzare modelli pre-addestrati condivisi
import tensorflow_datasets as tfds   # TensorFlow Datasets: per accedere a dataset comuni pre-configurati

# Versioni richieste
# Qui definiamo le versioni specifiche delle librerie necessarie per garantire compatibilità e riproducibilità
tf_version = "2.18.0"                # Versione di TensorFlow desiderata
hub_version = "0.16.1"               # Versione di TensorFlow Hub desiderata
datasets_version = "4.9.8"           # Versione di TensorFlow Datasets desiderata

# Controlliamo le versioni dei pacchetti e agiamo di conseguenza
if (tf.__version__ != tf_version or  # Verifica se la versione corrente di TensorFlow è diversa da quella richiesta
    hub.__version__ != hub_version or  # Verifica la versione di TensorFlow Hub
    tfds.__version__ != datasets_version):  # Verifica la versione di TensorFlow Datasets

    # Notifica all'utente le versioni attuali e quelle che verranno installate
    print(f"Versione TensorFlow corrente: {tf.__version__}, switching a {tf_version}")
    print(f"Versione TensorFlow Hub corrente: {hub.__version__}, switching a {hub_version}")
    print(f"Versione TensorFlow Datasets corrente: {tfds.__version__}, switching a {datasets_version}")

    # Disinstalliamo le versioni correnti dei pacchetti (se necessario)
    !pip uninstall -y tensorflow tensorflow_hub tensorflow_datasets

    # Installiamo le versioni richieste dei pacchetti specificati
    !pip install tensorflow==2.18.0 tensorflow_hub==0.16.1 tensorflow_datasets==4.9.8

    # Informiamo l'utente che è necessario riavviare il runtime per applicare i cambiamenti
    print("Le versioni specificate di TensorFlow, TensorFlow Hub e TensorFlow Datasets sono state installate.")
    print("Prego cliccare su Runtime > Restart session and run all.")
else:
    # Se tutte le versioni corrispondono a quelle richieste, lo segnaliamo all'utente
    print("Tutti i pacchetti sono nelle versioni specificate")


In [ ]:
# 2) Dataset e preprocessing
# Importiamo le librerie restanti necessarie
import numpy as np                      # Per operazioni matematiche e manipolazione di array
import matplotlib.pylab as plt          # Per la visualizzazione dei dati

# Funzione per formattare le immagini
def format_image(image, label):
    """
    Ridimensiona l'immagine a 224x224 pixel e normalizza i valori dei pixel tra 0 e 1.
    Args:
        image: Tensor dell'immagine originale
        label: Etichetta associata all'immagine
    Returns:
        L'immagine ridimensionata e normalizzata, insieme alla sua etichetta
    """
    image = tf.image.resize(image, (224, 224)) / 255.0  # Ridimensiona e normalizza
    return image, label

# Caricamento del dataset 'cats_vs_dogs' da TensorFlow Datasets
# Dividiamo il dataset in 3 sottoinsiemi: addestramento, validazione e test
(raw_train, raw_validation, raw_test), metadata = tfds.load(
    'cats_vs_dogs',                         # Nome del dataset
    split=['train[:80%]', 'train[80%:90%]', 'train[90%:]'],  # Suddivisione personalizzata
    with_info=True,                         # Carica anche informazioni aggiuntive sul dataset
    as_supervised=True,                     # Ritorna coppie (immagine, etichetta)
)

# Mostriamo alcune informazioni sul dataset
num_examples = metadata.splits['train'].num_examples  # Numero totale di esempi nel dataset di addestramento
num_classes = metadata.features['label'].num_classes  # Numero di classi (in questo caso, 2: gatti e cani)
print(num_examples)  # Stampa il numero di esempi
print(num_classes)   # Stampa il numero di classi

# Preparazione dei batch di dati per addestramento, validazione e test
BATCH_SIZE = 32  # Dimensione dei batch

# Creiamo i batch per l'addestramento: mescoliamo, formattiamo e prefetch per ottimizzare le prestazioni
train_batches = raw_train.shuffle(num_examples // 4).map(format_image).batch(BATCH_SIZE).prefetch(1)

# Creiamo i batch per la validazione: formattiamo e utilizziamo batch di dimensione definita
validation_batches = raw_validation.map(format_image).batch(BATCH_SIZE).prefetch(1)

# Creiamo i batch per il test: formattiamo le immagini ma usiamo batch di dimensione 1
test_batches = raw_test.map(format_image).batch(1)

# Visualizzazione della forma dei dati
# Prendiamo un batch dal dataset di addestramento per verificare le dimensioni delle immagini e delle etichette
for image_batch, label_batch in train_batches.take(1):  # Estraiamo un batch dal dataset di addestramento
    pass
image_batch.shape  # Mostriamo la forma del batch di immagini

In [ ]:
# 3) Configurazione del modello pre-addestrato

# URL del modello MobileNetV2 senza testa (solo feature vector), ospitato su TensorFlow Hub.
module_handle = "https://tfhub.dev/google/tf2-preview/mobilenet_v2/feature_vector/4"
# Dimensione di input delle immagini. MobileNetV2 richiede immagini 224x224
IMAGE_SIZE = (224, 224)
# Dimensione del vettore di caratteristiche (feature vector) prodotto da MobileNetV2. È una rappresentazione numerica dell'immagine
FV_SIZE = 1280
num_classes = 2 # Numero di classi da classificare, in questo caso 2 (es. cani e gatti).

# Carica il modulo da TF Hub
# Carica il modello pre-addestrato da TensorFlow Hub. feature_extractor sarà una funzione che prende un'immagine e restituisce un vettore di caratteristiche
feature_extractor = hub.load(module_handle)

# Definisci il modello Keras usando l'API funzionale
# Definisce l'input del modello Keras. Le immagini hanno dimensione (224, 224, 3), dove 3 sono i canali RGB.
inputs = tf.keras.Input(shape=IMAGE_SIZE + (3,))

# Crea un layer Lambda che applica la funzione feature_extractor all'input.
# L'uso di Lambda è utile per evitare errori quando si integra un modello TF Hub in un modello Keras.
feature_vector = tf.keras.layers.Lambda(
    lambda x: feature_extractor(x), output_shape=(FV_SIZE,), trainable=False
)(inputs)

# Aggiunge un layer denso (Dense) finale con num_classes neuroni e attivazione softmax, per ottenere una distribuzione di probabilità sulle classi.
dense_output = tf.keras.layers.Dense(num_classes, activation='softmax')(feature_vector)

# Definisci il modello completo specificando input e output
model = tf.keras.Model(inputs=inputs, outputs=dense_output)

# Stampa un riassunto della struttura del modello: layer, dimensioni in input/output e numero di parametri.
model.summary()

model.compile(                                  # Compila il modello specificando:
    optimizer='adam',                           # Ottimizzatore: adam, adatto alla maggior parte dei casi.
    loss='sparse_categorical_crossentropy',     # Funzione di perdita: sparse_categorical_crossentropy, usata per classificazione multiclasse con etichette intere (es. 0, 1).
    metrics=['accuracy']                        # Metriche: accuracy per valutare la performance.
)

### Quindi alleniamo e salviamo il nostro modello
Poiché stiamo eseguendo il transfer learning per mettere a punto un modello pre-addestrato sul nostro dataset, possiamo usare solo 5 Epoch.

In [ ]:
# 4) Addestramento Specifico con il dataset cat_vs_dog
EPOCHS = 5                                              # Definisce il numero di epoche (cicli completi sul dataset di addestramento).
# Addestra il modello usando il metodo fit di Keras.
hist = model.fit(train_batches,                         # train_batches: è il dataset di addestramento contenente immagini e relative etichette.
                 epochs=EPOCHS,                         # epochs=EPOCHS: specifica per quante epoche addestrare (5 in questo caso).
                 validation_data=validation_batches)    # fornisce un dataset di validazione. Viene usato per valutare la performance del modello su dati non visti
                                                        # dopo ogni epoca (utile per controllare overfitting).

In [ ]:
# 5) Salvataggio del Modello "specializzato"
import pathlib                                # pathlib per operazioni su percorsi di file
# Esportazione del modello in formato SavedModel
CATS_VS_DOGS_SAVED_MODEL = 'saved_model/1'                     # Directory di destinazione
model.export(CATS_VS_DOGS_SAVED_MODEL)                         # Esporta il modello in SavedModel compatibile TF

# Calcolo dell’occupazione di memoria del modello salvato
import os                                       #libreria per la gestione dei path di disco
def get_directory_size(CATS_VS_DOGS_SAVED_MODEL):
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(CATS_VS_DOGS_SAVED_MODEL):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total_size += os.path.getsize(fp)
#    return total_size / (1024 * 1024)          # Dimensione in MB
    return total_size                           # Dimensione in Byte

file_size_savedmodel = get_directory_size(CATS_VS_DOGS_SAVED_MODEL)
#print(f"Dimensione del modello SavedModel: {file_size_savedmodel:.2f} MB")
print(f"Dimensione del modello SavedModel: {file_size_savedmodel:.2f} Byte")


### FASE DI RIDUZIONE DEL MODELLO:


In [ ]:
# 6) FASE DI RIDUZIONE (Tre tipologie di riduzione a confronto)

# Configurazione del converter TFLite per il SavedModel
converter = tf.lite.TFLiteConverter.from_saved_model(CATS_VS_DOGS_SAVED_MODEL)  # Crea il converter
converter.experimental_enable_resource_variables = True          # Fold delle variabili in costanti

# Creazione di un convertitore TFLite dal modello salvato
converter = tf.lite.TFLiteConverter.from_saved_model(CATS_VS_DOGS_SAVED_MODEL)

# Abilita le ottimizzazioni predefinite per ridurre le dimensioni e migliorare l'efficienza
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # Decommenta questa linea per Model 2 e Model 3

# Generazione di un dataset rappresentativo per la quantizzazione intera (da usare per il Model 3)
# Decommenta queste 5 linee sotto per ottenere il Model 3
def representative_data_gen():
  for input_value, _ in test_batches.take(100):  # Usa 100 batch di dati di test per rappresentare la distribuzione
    yield [input_value]
converter.representative_dataset = representative_data_gen  # Assegna il dataset rappresentativo
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8] # Specifica di supportare operazioni quantizzate (INT8)

# Conversione del modello in formato TFLite
tflite_model = converter.convert()

# Specifica la directory per salvare il modello TFLite
tflite_models_dir = pathlib.Path("/tmp/")  # Usa una directory temporanea

# Salva il modello convertito in un file chiamato "model1.tflite", "model2.tflite", oppure "model3.tflite"
tflite_model_file = tflite_models_dir / 'model3.tflite'   # Nome del file per il modello TFLite da modificare in model2 e model3
tflite_model_file.write_bytes(tflite_model)               # Scrive il contenuto del modello TFLite nel file

In [ ]:
# 7) Test con Validazione del modello esaminato (model1, model2 o model3)
#@title Esegui questa cella ogni volta per testare l'accuratezza del tuo modello (assicurati di cambiare il nome del file)

from tqdm import tqdm  # Libreria per visualizzare barre di progresso
tflite_model_file = '/tmp/model3.tflite'  # Specifica il percorso ed il nome del file con cui è stato salvato il modello TFLite
interpreter = tf.lite.Interpreter(model_path=tflite_model_file)  # Carica il modello TFLite e Istanzia un interprete TFLite per eseguire l'inferenza.
interpreter.allocate_tensors()  # Alloca i tensori necessari per l'inferenza

input_index = interpreter.get_input_details()[0]["index"]  # Ottiene l'indice del tensore di input
output_index = interpreter.get_output_details()[0]["index"]  # Ottiene l'indice del tensore di output

predictions = []  # Array per salvare le predizioni
test_labels, test_imgs = [], []  # Array per etichette e immagini di test

for img, label in tqdm(test_batches.take(100)):  # ciclo di for su un gruppo di 100 immagini di test con le relative etichette
    interpreter.set_tensor(input_index, img)  # Imposta il tensore di input con l'immagine
    interpreter.invoke()  # Esegue l'inferenza
    predictions.append(interpreter.get_tensor(output_index))  # Salva la predizione
    test_labels.append(label.numpy()[0])  # Salva l'etichetta reale
    test_imgs.append(img)  # Salva l'immagine

# Nota: le iterazioni al secondo e quindi la velocità di esecuzione dell'inferenza dipende dal dispositivo su cui si fa girare il modello

score = 0  # Contatore per il numero di predizioni corrette

for item in range(0,100):  # Itera su tutte le predizioni
  prediction = np.argmax(predictions[item])  # Seleziona l'indice con il valore più alto nella distribuzione di probabilità predetta e converte la predizione in una etichetta
  label = test_labels[item]  # Etichetta reale
  if prediction == label:  # Confronta predizione ed etichetta reale
    score += 1  # Incrementa il punteggio in caso di successo

print("Su 100 previsioni che ho ricevuto " + str(score) + " corretti")

In [ ]:
# 8) Rappresentazione grafica
#@title Funzione di utilità per la rappresentazione grafica

class_names = ['cat', 'dog']  # Array dei nomi delle classi per la classificazione (usati per visualizzare i risultati in modo leggibile)

def plot_image(i, predictions_array, true_label, img):  # Definizione della funzione che mostra l'immagine e i risultati della predizione
    predictions_array, true_label, img = predictions_array[i], true_label[i], img[i]  # Estrai i dati relativi all'indice `i`: la distribuzione di probabilità, l'etichetta reale e l'immagine
    plt.grid(False)  # Disabilita la griglia sul grafico per una visualizzazione più pulita
    plt.xticks([])  # Rimuove i segni sull'asse x
    plt.yticks([])  # Rimuove i segni sull'asse y

    img = np.squeeze(img)  # Rimuove informazioni inutili per l'immagine per facilitare la visualizzazione
    plt.imshow(img, cmap=plt.cm.binary)  # Mostra l'immagine usando una mappa colori in scala di grigi (utile per immagini normalizzate)

    predicted_label = np.argmax(predictions_array)  # Ottiene l'indice della classe con la probabilità massima dalla distribuzione di probabilità predetta

    if predicted_label == true_label:  # Controlla se la classe predetta corrisponde all'etichetta reale
        color = 'green'  # Usa il colore verde per il testo se la predizione è corretta
    else:
        color = 'red'  # Usa il colore rosso per il testo se la predizione è errata

    plt.xlabel(  # Aggiunge un'etichetta sotto l'immagine per mostrare i dettagli della predizione
        "{} {:2.0f}% ({})".format(  # Formatta la stringa per includere il nome della classe predetta, la probabilità e la classe reale
            class_names[predicted_label],          # Nome della classe predetta
            100 * np.max(predictions_array),      # La probabilità della classe predetta in percentuale
            class_names[true_label]               # Nome della classe reale
        ),
        color=color  # Imposta il colore del testo basato sulla correttezza della predizione
    )

In [ ]:
# 9) Visualizzazione delle inferenze con la relativa classificazione
#@title Visualizza gli output ogni volta { run: "auto" }
# Questa riga è un'annotazione di Colab che consente di creare un widget interattivo
# per controllare il codice (in questo caso un cursore per scegliere `max_index`).

max_index = 10 #@param {type:"slider", min:1, max:100, step:1}
# Questa è un'altra annotazione di Colab: crea uno slider interattivo che
# permette di scegliere un valore intero tra 1 e 100.

for index in range(0, max_index):  # Ciclo che itera sugli indici delle immagini
                                   # da 0 fino a `max_index - 1`
    plt.figure(figsize=(6,3))  # Crea una nuova figura con dimensioni di 6x3 pollici

    plt.subplot(1, 2, 1)  # Divide la figura in una griglia di 1 riga e 2 colonne, selezionando la prima sezione
    plot_image(index, predictions, test_labels, test_imgs)  # Chiama la funzione `plot_image` per visualizzare l'immagine e i dettagli della predizione

    plt.show()  # Mostra il grafico. Questo comando assicura che ogni immagine venga visualizzata separatamente.